# NEEZY cloud training and candidate publishing

This notebook runs **data collection, training, and held-out trade backtests only**. The always-on Windows PC remains the lightweight MT5 executor. Kaggle and Colab sessions expire, so nothing here places trades or pretends to run 24/7.

A successful run uploads a model candidate, its validation manifest, and the public-candle dataset to Hugging Face Hub. It deliberately **does not update `latest/`**. Exact-hash paper-trading evidence and a separate promotion command are required before a PC can activate the candidate.

## 1. Settings and installation

Before running, add these secrets in **Kaggle → Add-ons → Secrets** or **Colab → Secrets**:

- `HF_TOKEN`: a fine-grained token with write access to the two repositories below;
- `HF_MODEL_REPO`: for example `your-user/neezy-models`;
- `HF_DATASET_REPO`: for example `your-user/neezy-datasets`.

Create both repositories first (model and dataset types respectively). Never paste the token into a cell or save it in notebook output. The current compact JSON model trains on CPU; do not spend GPU quota unless you replace it with a model that actually uses the GPU.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

GITHUB_REPO = "jvrboy/NEEZY"
GIT_REF = "main"  # use an immutable release tag for reproducible runs
SYMBOLS = ["frxEURUSD"]
TIMEFRAME = "15m"
CANDLES = 5000
EPOCHS = 60
SPREAD = 0.0  # full spread in price units; set a realistic demo-broker assumption

WORK = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("/content")
if not WORK.exists():
    WORK = Path.home() / "neezy-cloud"
WORK.mkdir(parents=True, exist_ok=True)
SOURCE = WORK / "neezy-source"
if not (SOURCE / ".git").exists():
    subprocess.run(
        [
            "git",
            "clone",
            "--depth",
            "1",
            "--branch",
            GIT_REF,
            f"https://github.com/{GITHUB_REPO}.git",
            str(SOURCE),
        ],
        check=True,
    )
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        "-r",
        str(SOURCE / "requirements-training.txt"),
    ],
    check=True,
)
os.chdir(SOURCE)
print(f"Source: {SOURCE}; heavy work stays in this temporary cloud session.")

In [ ]:
# Load secrets without printing their values.
secret_values = {}
try:
    from kaggle_secrets import UserSecretsClient

    secrets_client = UserSecretsClient()
    for name in ("HF_TOKEN", "HF_MODEL_REPO", "HF_DATASET_REPO"):
        try:
            secret_values[name] = secrets_client.get_secret(name)
        except Exception:  # noqa: BLE001
            pass
except ImportError:
    try:
        from google.colab import userdata

        for name in ("HF_TOKEN", "HF_MODEL_REPO", "HF_DATASET_REPO"):
            try:
                secret_values[name] = userdata.get(name)
            except Exception:  # noqa: BLE001
                pass
    except ImportError:
        pass

for name, value in secret_values.items():
    if value:
        os.environ[name] = value
missing = [
    name
    for name in ("HF_TOKEN", "HF_MODEL_REPO", "HF_DATASET_REPO")
    if not os.environ.get(name)
]
if missing:
    raise RuntimeError(f"Add these platform secrets, then rerun: {', '.join(missing)}")
print("Hugging Face credentials loaded from platform secrets (values hidden).")

## 2. Train, backtest, and upload candidates

The trade backtest uses only the model's chronological held-out slice, fills at the next candle open, and resolves ambiguous stop/target candles stop-first. A candidate can still fail the gate; it remains available for research but cannot be promoted until every PC policy check passes.

In [ ]:
command = [
    sys.executable,
    "scripts/cloud_train.py",
    "--symbols",
    *SYMBOLS,
    "--timeframe",
    TIMEFRAME,
    "--candles",
    str(CANDLES),
    "--epochs",
    str(EPOCHS),
    "--spread",
    str(SPREAD),
    "--upload",
    "--output",
    str(WORK / "candidates"),
]
subprocess.run(command, check=True)

## 3. Paper-trade the exact candidate, then promote deliberately

Keep `latest/` unchanged. On the Windows PC, use a real MT5 demo connection and set this no-order mode in `executor.json` (the spread is in price units, not pips):

```json
{"execution_enabled": false, "paper_candidate": "manifests/MODEL_ID.json", "paper_spread": 0.0001}
```

The lightweight service validates the candidate, records persistent virtual trades, and writes an exact-hash report under `NEEZY_HOME/paper-reports/`. It never calls broker order methods. Defaults require 14 full days, 20 closed virtual trades, and positive spread-adjusted performance. See `docs/CLOUD_OPERATIONS.md` for gap recovery and report handling.

From a shell with the training dependencies and `HF_TOKEN` set, perform a dry run first, then add `--promote` only if every check passes:

```bash
python scripts/promote_model.py --hf-model-repo YOUR_USER/neezy-models \
  --candidate manifests/MODEL_ID.json --paper-report paper-report.json
```

The PC checks `latest/` every three hours by default. Hash mismatches, weak held-out metrics, a failed backtest, missing paper evidence, or an incompatible schema reject the candidate and preserve the active model. Historical and paper results do not guarantee future performance.

In [ ]:
# List artifact names only; never print environment variables or secrets.
artifacts = sorted((WORK / "candidates").glob("*/manifest.json"))
for manifest in artifacts:
    print(manifest.parent.name)
print(f"{len(artifacts)} local candidate manifest(s); Hub latest was not modified.")